# E07 — quantas explicações cabem nos mesmos dados

O capítulo 7 fechou em pergunta de informação: quanta cabe num pedaço finito de passado, e a
partir de quando duas explicações ficam indistinguíveis. Este caderno torna a pergunta contável.

**A conta.** Uma família de explicações --- duas leis, uma calma e uma agitada ---, e quatro
estatísticas medidas no dado. Quantos membros da família reproduzem **todas** elas dentro da
tolerância?

**Os dois desfechos que não podem ser confundidos.** Uma estatística pode deixar uma **região**
(muitas explicações cabem, e o dado não escolhe entre elas) ou cortar o conjunto **a zero**
(nenhuma cabe, e a família inteira é refutada). Dizer "o modelo explica os dados" sem dizer qual
dos dois é o defeito que o módulo regimes.py torna impossível.

**O controle da tolerância.** A tolerância foi escolhida por mim, e escolha de quem mede contamina
a medida: o caderno repete a contagem com ela apertada e com ela frouxa, e se o veredito mudar
então o veredito é meu e não do dado.

**Convenções** (AGENTS.md §7 e §9): parâmetros no topo marcados "brinque com", algoritmo em
frevolab, resultado em lab/resultados/E07_condicionamento.json.

In [1]:
# <- brinque com: SERIE, GRADE_P, GRADE_RAZAO, GRADE_PERMANENCIA, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, regimes, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"
JANELA, POSTO, BLOCO = 252, 13, 60
GRADE_P = (0.02, 0.05, 0.08, 0.12, 0.18, 0.25)
GRADE_RAZAO = (1.5, 2.0, 2.5, 3.0, 3.5, 4.0)
GRADE_PERMANENCIA = (1.0, 10.0, 30.0, 60.0, 120.0)
TOLERANCIA = {"taxa": 0.002, "pior": 2.0, "mediana": 1.0, "acima_do_dobro": 0.03}
SEMENTE = 131
SEMENTES = 10    # o controle da semente: a contagem publicada e de uma so

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
x = retornos.to_numpy()
sigma = float(x.std(ddof=1))
print("frevolab %s | %s: %d dias | desvio diario %.5f" % (frevolab.VERSAO, SERIE, len(x), sigma))

frevolab 0.1.0 | sp500.csv: 6718 dias | desvio diario 0.01213


## O que o dado mede

In [2]:
# O que o dado mede: a taxa do capítulo 3 e as tres leituras da forma.
real = regimes.estatisticas(x, JANELA, POSTO, BLOCO)
print("taxa %.4f | pior bloco %d | mediana %.0f | blocos acima do dobro %.4f"
      % (real["taxa"], real["pior"], real["mediana"], real["acima_do_dobro"]))

taxa 0.0513 | pior bloco 20 | mediana 2 | blocos acima do dobro 0.1305


## Sem memória: muitas explicações cabem

In [3]:
# Sem memoria: o estado e sorteado de novo a cada dia.
sorteio = np.random.default_rng(SEMENTE)
sem_memoria = []
for p in GRADE_P:
    for razao in GRADE_RAZAO:
        e = regimes.estatisticas(regimes.mistura(len(x), sorteio, sigma, p, razao), JANELA, POSTO, BLOCO)
        sem_memoria.append({"p": p, "razao": razao, "estatisticas": e})
print("combinacoes: %d" % len(sem_memoria))
for chaves in (("taxa",), ("taxa", "pior"), ("taxa", "pior", "mediana"),
               ("taxa", "pior", "mediana", "acima_do_dobro")):
    print("   %-46s %3d cabem" % (" + ".join(chaves), len(regimes.cabem(sem_memoria, real, TOLERANCIA, chaves))))
piores = [c["estatisticas"]["pior"] for c in sem_memoria]
print("   o melhor pior bloco da familia sem memoria: %d (o dado tem %d)" % (max(piores), real["pior"]))

combinacoes: 36
   taxa                                            34 cabem
   taxa + pior                                      0 cabem
   taxa + pior + mediana                            0 cabem
   taxa + pior + mediana + acima_do_dobro           0 cabem
   o melhor pior bloco da familia sem memoria: 11 (o dado tem 20)


## Com persistência: a família chega perto

In [4]:
# Com persistencia: o estado dura, e a duracao media e um parametro a mais.
persistente = []
for p in GRADE_P:
    for razao in GRADE_RAZAO:
        for permanencia in GRADE_PERMANENCIA:
            e = regimes.estatisticas(
                regimes.persistente(len(x), sorteio, sigma, p, razao, permanencia), JANELA, POSTO, BLOCO)
            persistente.append({"p": p, "razao": razao, "permanencia": permanencia, "estatisticas": e})
print("combinacoes: %d" % len(persistente))
for chaves in (("taxa",), ("taxa", "pior"), ("taxa", "pior", "mediana"),
               ("taxa", "pior", "mediana", "acima_do_dobro")):
    c = regimes.cabem(persistente, real, TOLERANCIA, chaves)
    print("   %-46s %3d cabem%s" % (" + ".join(chaves), len(c),
          (" | permanencias: %s" % sorted({y["permanencia"] for y in c})) if 0 < len(c) < 40 else ""))
piores_p = [(c["estatisticas"]["pior"], c["permanencia"]) for c in persistente]
excessos = [(c["estatisticas"]["acima_do_dobro"], c["permanencia"]) for c in persistente]
print("   melhor pior bloco com persistencia: %d (permanencia %s) | o dado tem %d"
      % (max(piores_p)[0], max(piores_p)[1], real["pior"]))
print("   maior excesso de blocos cheios: %.4f (permanencia %s) | o dado tem %.4f"
      % (max(excessos)[0], max(excessos)[1], real["acima_do_dobro"]))

combinacoes: 180
   taxa                                           122 cabem
   taxa + pior                                     12 cabem | permanencias: [10.0, 30.0, 60.0, 120.0]
   taxa + pior + mediana                           12 cabem | permanencias: [10.0, 30.0, 60.0, 120.0]
   taxa + pior + mediana + acima_do_dobro           2 cabem | permanencias: [10.0, 30.0]
   melhor pior bloco com persistencia: 23 (permanencia 30.0) | o dado tem 20
   maior excesso de blocos cheios: 0.2249 (permanencia 60.0) | o dado tem 0.1305


In [5]:
# O controle da semente. A pergunta "quantos membros da familia cabem?" e respondida por UMA
# varredura, e o veredito depende de quais mundos foram sorteados: o capitulo 9 publica a sua
# contagem e o capitulo 10 publica outra para a mesma serie e a mesma tolerancia. A mesma
# varredura em dez sementes diz quanto da resposta e do dado e quanto e do sorteio.
quatro = ("taxa", "pior", "mediana", "acima_do_dobro")
contagens = []
for s in range(SEMENTE, SEMENTE + SEMENTES):
    sorteio_s = np.random.default_rng(s)
    candidatos = []
    for p in GRADE_P:
        for razao in GRADE_RAZAO:
            for permanencia in GRADE_PERMANENCIA:
                e = regimes.estatisticas(
                    regimes.persistente(len(x), sorteio_s, sigma, p, razao, permanencia),
                    JANELA, POSTO, BLOCO)
                candidatos.append({"p": p, "razao": razao, "permanencia": permanencia,
                                   "estatisticas": e})
    contagens.append(len(regimes.cabem(candidatos, real, TOLERANCIA, quatro)))
print("contagem por semente: %s" % contagens)
print("menor %d | mediana %.1f | maior %d | sementes %d"
      % (min(contagens), float(np.median(contagens)), max(contagens), len(contagens)))


contagem por semente: [2, 2, 4, 5, 2, 3, 3, 1, 0, 3]
menor 0 | mediana 2.5 | maior 5 | sementes 10


## O controle da tolerância

In [6]:
# O controle da tolerancia: se o veredito mudar ao aperta-la, ele e meu e nao do dado.
quatro = ("taxa", "pior", "mediana", "acima_do_dobro")
linhas = []
for nome, fator in (("apertada", 0.5), ("a escolhida", 1.0), ("frouxa", 2.0)):
    tol = {k: v * fator for k, v in TOLERANCIA.items()}
    linhas.append({
        "tolerancia": nome,
        "sem memoria: taxa": len(regimes.cabem(sem_memoria, real, tol, ("taxa",))),
        "sem memoria: taxa + pior": len(regimes.cabem(sem_memoria, real, tol, ("taxa", "pior"))),
        "persistente: taxa": len(regimes.cabem(persistente, real, tol, ("taxa",))),
        "persistente: as quatro": len(regimes.cabem(persistente, real, tol, quatro)),
    })
print(pd.DataFrame(linhas).set_index("tolerancia").to_string())

             sem memoria: taxa  sem memoria: taxa + pior  persistente: taxa  persistente: as quatro
tolerancia                                                                                         
apertada                    24                         0                 73                       1
a escolhida                 34                         0                122                       2
frouxa                      36                         0                150                      24


## As figuras

In [7]:
# Figura 1: ate onde cada familia alcanca o agrupamento do dado.
fig, eixo = plt.subplots(figsize=(9.4, 4.4))
limites = np.arange(6, 24)
eixo.hist([c["estatisticas"]["pior"] for c in sem_memoria], bins=limites, color="#b03a2e",
          alpha=0.75, label="sem memória")
eixo.hist([c["estatisticas"]["pior"] for c in persistente], bins=limites, color="#1f4e79",
          alpha=0.75, label="com persistência")
eixo.axvline(real["pior"], color="#333333", ls="--", lw=1.6,
             label="o dado: %d rompimentos no pior bloco" % real["pior"])
eixo.set_xlabel("pior bloco de %d dias" % BLOCO)
eixo.set_ylabel("explicações da família")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E07_condicionamento", 1)
plt.close(fig)
print("pior bloco: sem memoria ate %d | com persistencia ate %d | o dado %d"
      % (max(c["estatisticas"]["pior"] for c in sem_memoria),
         max(c["estatisticas"]["pior"] for c in persistente), real["pior"]))

pior bloco: sem memoria ate 11 | com persistencia ate 23 | o dado 20


In [8]:
# Figura 2: o conjunto factivel no plano (p, razao), por quantas estatisticas cada ponto satisfaz.
malha_taxa = np.zeros((len(GRADE_P), len(GRADE_RAZAO)))
malha_quatro = np.zeros((len(GRADE_P), len(GRADE_RAZAO)))
for c in persistente:
    if c["permanencia"] != max(GRADE_PERMANENCIA):
        continue
    i, j = GRADE_P.index(c["p"]), GRADE_RAZAO.index(c["razao"])
    malha_taxa[i, j] = sum(abs(c["estatisticas"][k] - real[k]) <= TOLERANCIA[k] for k in ("taxa",))
    malha_quatro[i, j] = sum(abs(c["estatisticas"][k] - real[k]) <= TOLERANCIA[k] for k in quatro)
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.2))
for eixo, malha, titulo in ((esq, malha_taxa, "so a taxa"), (dir_, malha_quatro, "as quatro")):
    imagem = eixo.imshow(malha, origin="lower", cmap="Blues", vmin=0, vmax=4, aspect="auto")
    eixo.set_xticks(range(len(GRADE_RAZAO)))
    eixo.set_xticklabels(["%.1f" % r for r in GRADE_RAZAO], fontsize=8)
    eixo.set_yticks(range(len(GRADE_P)))
    eixo.set_yticklabels(["%.2f" % p for p in GRADE_P], fontsize=8)
    eixo.set_xlabel("razão entre as duas leis")
    eixo.set_ylabel("p, a fração agitada")
    eixo.set_title(titulo, fontsize=10)
fig.colorbar(imagem, ax=(esq, dir_), shrink=0.8, label="estatísticas satisfeitas")
fig.tight_layout()
graficos.salvar(fig, "E07_condicionamento", 2)
plt.close(fig)
print("pontos que satisfazem so a taxa: %d de %d | as quatro: %d"
      % (int(malha_taxa.sum()), malha_taxa.size, int((malha_quatro == 4).sum())))

/tmp/ipykernel_2778154/1931979498.py:21: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


pontos que satisfazem so a taxa: 14 de 36 | as quatro: 0


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Dois histogramas sobrepostos, um vermelho e um azul, e a linha tracejada do dado no
extremo direito. O vermelho cobre uma faixa estreita à esquerda e para bem antes da linha; o azul
avança até encostar nela. O que a figura engana: com as barras sobrepostas as cores ficam turvas e
não se sabe qual massa é qual, e a linha do dado, colada na borda do painel, faz o vazio entre as
duas famílias parecer uma margem apertada — quando é um muro.

**Figura 2.** Dois mapas lado a lado com os mesmos eixos (a razão entre as leis, e a fração
agitada). O da esquerda é quase todo claro; o da direita é escuro em quase tudo, com uma faixa
estreita que não é. O que a escala engana, e é o perigo desta figura: a cor conta **quantas
estatísticas** foram satisfeitas, e não quão bom é o ajuste — claro lê-se como aprovação, e um
leitor apressado pode tomar a mancha clara da esquerda como "o modelo funciona".


In [9]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "condicionamento_dias": int(len(x)),
    "condicionamento_taxa": real["taxa"],
    "condicionamento_pior": real["pior"],
    "condicionamento_mediana": real["mediana"],
    "condicionamento_acima_do_dobro": real["acima_do_dobro"],
    "condicionamento_combinacoes": len(sem_memoria),
    "condicionamento_cabem_taxa": len(regimes.cabem(sem_memoria, real, TOLERANCIA, ("taxa",))),
    "condicionamento_cabem_taxa_pior": len(regimes.cabem(sem_memoria, real, TOLERANCIA, ("taxa", "pior"))),
    "condicionamento_melhor_pior": int(max(piores)),
    "condicionamento_persistente_combinacoes": len(persistente),
    "condicionamento_persistente_cabem_taxa": len(regimes.cabem(persistente, real, TOLERANCIA, ("taxa",))),
    # A grandeza que o capítulo 8 imprimia por extenso ("cabem doze"): medida, e sem comando
    # ate aqui, de modo que a reescrita de um caderno podia desatualiza-la em silencio.
    "condicionamento_persistente_cabem_taxa_pior":
        len(regimes.cabem(persistente, real, TOLERANCIA, ("taxa", "pior"))),
    "condicionamento_persistente_cabem_quatro": len(regimes.cabem(persistente, real, TOLERANCIA, quatro)),
    "condicionamento_repeticoes": int(len(contagens)),
    "condicionamento_cabem_quatro_menor": int(min(contagens)),
    "condicionamento_cabem_quatro_mediana": float(np.median(contagens)),
    "condicionamento_cabem_quatro_maior": int(max(contagens)),
    "condicionamento_persistente_melhor_pior": int(max(piores_p)[0]),
    "condicionamento_persistente_melhor_pior_permanencia": float(max(piores_p)[1]),
    "condicionamento_persistente_melhor_excesso": float(max(excessos)[0]),
    "condicionamento_persistente_melhor_excesso_permanencia": float(max(excessos)[1]),
    "condicionamento_pontos_so_a_taxa": int(malha_taxa.sum()),
    "condicionamento_pontos_as_quatro": int((malha_quatro == 4).sum()),
}
for linha in linhas:
    nome = {"apertada": "apertada", "a escolhida": "escolhida", "frouxa": "frouxa"}[linha["tolerancia"]]
    resultado["condicionamento_%s_sem_memoria_taxa_pior" % nome] = linha["sem memoria: taxa + pior"]
    resultado["condicionamento_%s_persistente_quatro" % nome] = linha["persistente: as quatro"]


# O que sai do laboratorio e o que o livro cita: medida que o livro nao usa e medida morta.
CITADAS_NO_LIVRO = ("condicionamento_repeticoes", "condicionamento_cabem_quatro_menor",
                    "condicionamento_cabem_quatro_mediana", "condicionamento_cabem_quatro_maior","condicionamento_acima_do_dobro", "condicionamento_apertada_persistente_quatro", "condicionamento_apertada_sem_memoria_taxa_pior", "condicionamento_cabem_taxa", "condicionamento_cabem_taxa_pior", "condicionamento_combinacoes", "condicionamento_dias", "condicionamento_escolhida_persistente_quatro", "condicionamento_escolhida_sem_memoria_taxa_pior", "condicionamento_frouxa_persistente_quatro", "condicionamento_frouxa_sem_memoria_taxa_pior", "condicionamento_mediana", "condicionamento_melhor_pior", "condicionamento_persistente_cabem_quatro", "condicionamento_persistente_cabem_taxa",
    "condicionamento_persistente_cabem_taxa_pior", "condicionamento_persistente_cabem_taxa_pior",
    "condicionamento_persistente_cabem_taxa_pior", "condicionamento_persistente_combinacoes", "condicionamento_persistente_melhor_excesso", "condicionamento_persistente_melhor_pior", "condicionamento_pior", "condicionamento_pontos_as_quatro", "condicionamento_pontos_so_a_taxa", "condicionamento_taxa")
resultado = {chave: valor for chave, valor in resultado.items() if chave in CITADAS_NO_LIVRO}

caminho = Path("lab/resultados/E07_condicionamento.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E07_condicionamento.json gravado | 27 grandezas
